# Extraer ejercicios de formulación de emestrada

Lee los PDF de formulación de emestrada (`Formulación/{año} - Formulación.pdf`) y extrae, de cada página (una por examen):

- `exam_name`: nombre del examen (p. ej. `química - 2010 - junio`), que se usará como nombre de archivo.
- `statement`: enunciado de los compuestos a **nombrar**, que sirve de clave para hacer *match* con los ejercicios clasificados.

**Salida:** `csv/form_emestrada.csv`, que consume `match_nomenclature_exercises.ipynb`.

> Ejecutar con `notebooks/02_assign_exam_names` como directorio de trabajo.

In [23]:
import re
from pathlib import Path

import pandas as pd
import pdfplumber

In [34]:
ROOT = Path("~/code/selector_examenes").expanduser()
FORMULATION_DIR = ROOT / "PDF/Formulación"
OUTPUT_CSV = ROOT / "CSV/formulacion_emestrada.csv"
YEARS = range(2010, 2026)

## Lectura del PDF

La primera página es la portada; cada una de las siguientes contiene un ejercicio de un examen.

In [25]:
def extract_pages_text(file: Path) -> list[str]:
    """Devuelve el texto de cada página del PDF, omitiendo la portada."""
    with pdfplumber.open(file) as pdf:
        return [page.extract_text() for page in pdf.pages[1:]]

## Nombre del examen

El nombre aparece justo antes de la palabra «RESOLUCIÓN», salvo en 2023 y 2024, donde la página empieza por «R E S O L U C I Ó N» y el nombre está en la última línea.

In [26]:
RESOLUTION_PATTERN = re.compile(r"R\s*E\s*S\s*O\s*L\s*U\s*C\s*I\s*[ÓO]\s*N", re.IGNORECASE)


def extract_exam_name(page_text: str) -> str:
    """Extrae el nombre del examen (sin limpiar) del texto de una página."""
    lines = page_text.split("\n")

    # 2023 y 2024
    if lines[0] == "R E S O L U C I Ó N":
        return lines[-1]

    m = RESOLUTION_PATTERN.search(page_text)
    end_pos = m.start() if m else -1
    exam_name = page_text[:end_pos].strip().split("\n")[-1].lower()
    return re.sub(r"ejercicio \w+", "", exam_name)


def clean_exam_name(exam_name: str) -> str:
    """Normaliza el nombre del examen: 'Química 2010. Junio' -> 'química - 2010 - junio'."""
    exam_name = re.sub(r"\. ejercicio (?:A[1-2]|4A)", "", exam_name, flags=re.IGNORECASE)
    exam_name = re.sub(r"\. \. opción [ab]", "", exam_name)
    exam_name = re.sub(r"\s+", " ", exam_name)
    exam_name = exam_name.replace(".", " -").lower()
    return re.sub(r"- $", "", exam_name).strip()

## Enunciado de nomenclatura

- **Hasta 2024:** apartados a), b) y c) que siguen a «Formule o nombre los compuestos siguientes», hasta el apartado d).
- **2025:** solo el apartado c) de la línea de «Nombre o formule los siguientes compuestos» (compuesto orgánico).

In [27]:
STATEMENT_HEADER = re.compile(r"Formule o nombre los (?:siguientes compuestos|compuestos siguientes):? ")
STATEMENT_HEADER_2025 = "Nombre o formule los siguientes compuestos"


def _statement_until_2024(page_text: str, header: re.Match) -> str:
    start_pos = header.end()
    end_pos = page_text.find(" d)")
    return page_text[start_pos:end_pos]


def _statement_2025(page_text: str) -> str:
    lines = page_text.split("\n")
    header_idx = max(i for i, line in enumerate(lines) if line.startswith(STATEMENT_HEADER_2025))
    line = next(line for line in lines[header_idx:] if "c) " in line)
    return line[line.find("c) "):]


def extract_nomenclature_statement(page_text: str) -> str:
    """Extrae los compuestos a nombrar del enunciado, con espacios normalizados."""
    header = STATEMENT_HEADER.search(page_text)
    if header:
        statement = _statement_until_2024(page_text, header)
    else:
        statement = _statement_2025(page_text)

    statement = statement.replace(";", " ").strip()
    return re.sub(r"\s+", " ", statement)

## Procesar todos los PDF

In [28]:
def process_formulation_file(file: Path) -> list[dict]:
    """Devuelve un registro (año, examen, enunciado) por cada página del PDF."""
    year = int(file.name[:4])
    return [
        {
            "year": year,
            "exam_name": clean_exam_name(extract_exam_name(page_text)),
            "statement": extract_nomenclature_statement(page_text),
        }
        for page_text in extract_pages_text(file)
    ]

In [29]:
records = []
for year in YEARS:
    file = FORMULATION_DIR / f"{year} - Formulación.pdf"
    try:
        records.extend(process_formulation_file(file))
    except Exception as e:
        print(f"Año {year} ({file.name}). Error: {e!r}")

form_emestrada = pd.DataFrame(records)
form_emestrada.shape

(182, 3)

## Comprobaciones

In [30]:
# Ejercicios por año
form_emestrada.year.value_counts().sort_index()

year
2010    12
2011    12
2012    12
2013    12
2014    12
2015    12
2016    10
2017    12
2018    12
2019    12
2020    12
2021    12
2022    12
2023    12
2024    12
2025     4
Name: count, dtype: int64

In [31]:
# Años cuyos nombres de examen no contienen ese año (p. ej. PDF duplicados o mal nombrados)
mismatch = form_emestrada[[str(y) not in name for y, name in zip(form_emestrada.year, form_emestrada.exam_name)]]
mismatch

,year,exam_name,statement


In [32]:
form_emestrada.sample(10)

,year,exam_name,statement
173,2024,química - 2024 - reserva 3,a) Óxido de aluminio b) Ácido cloroso c) But-2...
77,2016,química - 2016 - reserva 2,a) Óxido de plomo(IV) b) Ácido sulfuroso c) Et...
52,2014,química - 2014 - reserva 2,a) Hidróxido de estroncio b) Dicromato de bari...
138,2021,química - 2021 - reserva 4,a) Ácido hipocloroso b) Sulfuro de cadmio c) P...
7,2010,química - 2010 - reserva 3,a) Bromuro de hidrógeno b) Carbonato de calcio...
116,2019,química - 2019 - septiembre,a) Hidruro de plomo(IV) b) Bromato de aluminio...
120,2020,química - 2020 - reserva 1,a) Sulfito de potasio b) Peróxido de sodio c) ...
59,2014,química - 2014 - septiembre,a) Óxido de cobalto(III) b) Hidrogenosulfato d...
26,2012,quimica - 2012 - reserva 1,a) Ácido clórico b) Seleniuro de hidrógeno c) ...
53,2014,química - 2014 - reserva 2,a) Cromato de plata b) Hidrogenocarbonato de p...


## Guardar

In [35]:
OUTPUT_CSV.parent.mkdir(exist_ok=True)
form_emestrada.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")